<div style="text-align: center;">

# Bank Pipeline: Pillar 3 Compliance and ECB Indicators

</div>

This notebook shows the end-to-end workflow for requesting a bank portfolio risk analysis from the PhysRisk API and preparing the resulting Pillar 3 and ECB indicator outputs.

The workflow is:

1. Load API configuration and credentials from the local environment.
2. Load a portfolio asset file and select the assets to include in the run.
3. Inspect the portfolio composition with helper visualizations.
4. Build and submit a bank portfolio risk request.
5. Download and extract the generated pipeline outputs.
6. Visualize the returned portfolio-level risk metrics.

The example uses a subset of the local `portfolio_asset.txt` file as portfolio example. The same request structure can be extended to any portfolios when the input asset records contain the fields required by the bank pipeline.

## 1. Imports

The notebook uses `requests` to call the API, `pathlib` and `zipfile` to manage downloaded output files, and local Plotly helper functions to visualize portfolio inputs and pipeline results.

In [1]:
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
from zipfile import ZipFile

In [2]:
from auxiliary_functions.portfolio_asset_plotly_charts import show_all_portfolio_charts, load_portfolio_asset
from auxiliary_functions.portfolio_metrics_plotly_charts import show_all_portfolio_metric_charts

## 2. Load API Configuration

The API base URL and API key are read from `../.env`. The API key is sent in the `X-API-Key` header for each request made to the bank pipeline endpoints.

Expected environment variables:

- `ALPHA_KLIMA_API_KEY`: API key used to authenticate API calls.
- `ALPHA_KLIMA_API_BASE_URL`: base URL of the Alpha-Klima API, for example the host that exposes `/api/calculate_portfolio_risks` and `/retrieve_pipeline_results`.

In [3]:
_ = load_dotenv("../.env")
API_KEY = os.environ.get("ALPHA_KLIMA_API_KEY")
API_BASE = os.environ.get("ALPHA_KLIMA_API_BASE_URL", "").rstrip("/")
headers = {"X-API-Key": API_KEY, "Accept": "application/json"}

## 3. Load Portfolio Data

The bank pipeline expects a `portfolio_asset` list where each item describes one exposure and its relevant asset, financial, and collateral attributes.

This section loads the local portfolio asset file for the example run. In production workflows, this input is typically generated from a bank portfolio extract after applying the same schema and data-quality requirements.

In [4]:
PORTFOLIO_ASSET_PATH = Path.cwd().parent / "resources" / "example_bank_portfolio.json"
portfolio_asset = load_portfolio_asset(PORTFOLIO_ASSET_PATH)
len(portfolio_asset)

31

In [5]:
portfolio_asset = portfolio_asset[15:30]
len(portfolio_asset)

15

### Example Portfolio Asset Schema

The commented payload below illustrates the fields expected for an individual portfolio asset. Location, `asset_class`, and `type` identify the physical asset and vulnerability model, while exposure, collateral, maturity, stage, and sector fields support the bank portfolio metrics and regulatory outputs.

In [6]:
# portfolio_asset = [
#     {
#         "id": 1,
#         "asset_id": "1",
#         "name": "LogiNavex Infraestructura S.A.",
#         "asset_class": "RealEstateAsset",
#         "type": "Buildings/Commercial",
#         "latitude": 37.27,
#         "longitude": -5.92,
#         "address": "Avenida de la Gran V\u00eda de Hortaleza, 3,  28033, Madrid",
#         "CNAE": 6820,
#         "group_CNAE": "L",
#         "debt_securities": 0,
#         "loans": 3000000.0,
#         "advances": 0,
#         "trade_finance": 0,
#         "equity": 0,
#         "maturity": "2035-02-02 00:00:00",
#         "stage": 2,
#         "physical_collateral": 3700000.0,
#         "collateral_lat": 39.3984615604245,
#         "collateral_lon": -0.393566078880586,
#         "financial_collateral": 0,
#         "gross_exposure": 3000000.0,
#         "tangible_to_total_ratio": 0.85,
#     }
# ]

### Inspect Portfolio Inputs

Before running the remote calculation, the notebook renders summary charts for the selected portfolio assets. These plots help verify that the sample contains the expected exposure mix, asset attributes, and geographic information.

In [7]:
figures = show_all_portfolio_charts(portfolio_asset)

## 4. Run Pillar 3 Analysis

This section submits the selected portfolio to `/api/calculate_portfolio_risks`. The endpoint runs the bank pipeline and generates portfolio risk outputs for the requested client and portfolio name.

The request combines:

- `client_id`: client namespace used by the API.
- `portfolio_assets`: the asset records loaded above.
- `portfolio_name`: label used to identify this pipeline run.

> **Note:** The example below uses `Arfima` as `client_id` for illustration. To run the pipeline against your own data, replace it with the tenant id registered for your account on the Alpha-Klima platform.

In [8]:
bank_request = {
    "client_id": "Arfima",
    "portfolio_assets": portfolio_asset,
    "portfolio_name": "prueba_bank_pipeline",
}


### Execute the Portfolio Risk Request

The request is sent with the API-key headers configured earlier. The returned response object is displayed first so the HTTP status can be checked before reading the JSON payload.

In [9]:
r = requests.post(
    url=f"{API_BASE}/api/calculate_portfolio_risks", json=bank_request, headers=headers
)
r

<Response [200]>

### Inspect the API Response

Printing the JSON response confirms whether the pipeline run was accepted and provides any status or diagnostic messages returned by the API.

In [10]:
print(r.json())

{'state': 'success', 'message': 'Portfolio risk calculation done'}


## 5. Download Results

After the portfolio risk calculation has completed, this section requests the generated result files from `/retrieve_pipeline_results` and writes the returned archive to `bank_results.zip`.

The selected outputs include exposure-level risk results, Pillar 3 template 5, portfolio risk metrics, and an automated Pillar 3 report.

In [11]:
download_request = {
    "client": "Arfima",
    "client_type": "bank",
    "files_to_download": [
        "risk_per_exposure",
        "pillar_3_template_5",
        "portfolio_risk_metrics",
        "pillar_3_automated_report",
    ],
}
response = requests.post(
    url=f"{API_BASE}/retrieve_pipeline_results", json=download_request, headers=headers
)
output_path = Path("bank_results.zip")
output_path.write_bytes(response.content)

21672

### Extract the Result Archive

The downloaded ZIP file is extracted into `downloaded_results`. The notebook then locates the generated results directory, which is used as the base path for reading the portfolio metric workbook in the final visualization step.

In [12]:
extract_dir = Path("downloaded_results")
extract_dir.mkdir(exist_ok=True)

with ZipFile(output_path) as zip_file:
    top_folders = {
        Path(name).parts[0]
        for name in zip_file.namelist()
        if len(Path(name).parts) > 1
    }

    if len(top_folders) != 1:
        raise ValueError(f"Expected one top-level folder, found: {top_folders}")

    subfolder_name = next(iter(top_folders))
    files_path = extract_dir / subfolder_name

    zip_file.extractall(extract_dir)

print(files_path)

downloaded_results\2026-05-28


## 6. Generated Output Files

The extracted results directory contains the main workbooks produced by the bank pipeline. Together, these files move from asset-level physical-risk evidence to portfolio-level indicators and regulatory reporting outputs.

- `Arfima_with_actives.xlsx`: asset and hazard summary. The sheet contains one row per asset-hazard-indicator-scenario-year combination, together with exposure attributes such as `gross_exposure`, `maturity`, `stage`, `group_cnae`, and `province_code`. The `risk_score` field summarizes the physical-risk level for that asset under the listed hazard context, so this file is useful for tracing which hazards drive risk at individual exposure level.
- `portfolio_metrics_Arfima.xlsx`: ECB indicator workbook. It includes portfolio risk-score distributions by hazard, overall portfolio metrics, and metrics by hazard. The risk-score distribution columns show the share of the portfolio in levels 0 to 3, from no risk to high risk. The metric sheets report `PEAR`, `NEAR`, and `CEAR`, which should be interpreted as portfolio-level climate-risk indicators and compared across hazards to identify the largest contributors.
- `template_5_Arfima.xlsx`: Pillar 3-compliant reporting template. The workbook contains the Annex XXXIX index and `5.CC Physical risk`, the Template 5 table for banking-book exposures subject to climate-change physical risk. Rows are organized by economic sector, while columns report gross carrying amount, maturity buckets, sensitivity to chronic and acute physical events, Stage 2 and non-performing exposures, and related impairment fields. This file is intended as the regulatory reporting view of the same pipeline results.

## 7. Visualize Portfolio Metrics

These figures summarize the bank pipeline outputs and provide a quick check that the downloaded result files are aligned with the submitted portfolio assets.

In [13]:
figures = show_all_portfolio_metric_charts(portfolio_asset, files_path / "portfolio_metrics_Arfima.xlsx")